Imports and environment

In [ ]:
# Install Python 3.10 or newer and run in your terminal:
# pip install "matplotlib>=3.0.0" "python-dateutil>=2.8.0" "python-dotenv>=0.21.0" "numpy==1.26.4" "opencv-python==4.11.0.86" "pandas==2.3.3" "pdf2image==1.17.0" "pillow==10.4.0" "PyMuPDF==1.26.7" "pytesseract>=0.3.10" "requests==2.32.5" "postal>=1.1.11" "geopy>=2.4.1"
# pip install ipykernel

# If using the LLMWhisperer option for new files, set up `LLMWHISPERER_API_KEY` in your environment. (Get free key online)

# If using tesseract, also need to install tesseract
# brew install tesseract (mac)
# sudo apt install tesseract-ocr libtesseract-dev tesseract-ocr-eng (linux)
# use windows installer from official repository

# if using Google geocoding API, set up `GOOGLE_GEOCODING_API_KEY` in environment

In [ ]:
import os
import json
import re
import time
import calendar
import string
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pymupdf as fitz
import pytesseract
import requests
import matplotlib.pyplot as plt
from pytesseract import Output
from PIL import Image
from pdf2image import convert_from_path
from dotenv import find_dotenv, load_dotenv
from dateutil import parser as date_parser
from postal.expand import expand_address
from postal.parser import parse_address
from geopy.geocoders import GoogleV3
from geopy.extra.rate_limiter import RateLimiter
from geopy.distance import geodesic

# API keys live in .env (LLMWHISPERER_API_KEY, GOOGLE_GEOCODING_API_KEY)
load_dotenv(find_dotenv(usecwd=True))

Constants and lookup tables

In [ ]:
DATA_ROOT = Path("data")
YEAR, REGION = 2024, "R5"
COUNTIES = ["fresno_madera", "kern", "kings", "tulare_west", "tulare_east", "rancho_cordova"]
FILES_CSV = Path("output_data/2024_files_by_template.csv")

LLMWHISPERER_BASE_URL = "https://llmwhisperer-api.us-central.unstract.com/api/v2"
LLMWHISPERER_HEADERS = {"unstract-key": os.getenv("LLMWHISPERER_API_KEY")}
PAGE_BREAK = "<<<PAGE_BREAK>>>"
PAGE_RE = re.compile(r"=== Page (\d+) ===")

# Terms found on manifest pages but nowhere else in a report's boilerplate
MANIFEST_TERMS = ["hauler info", "destination", "method used", "operator shall", "d-2",
                  "solids content", "hauler signature", "hauling event", "complete one"]
# Words on a manifest's second page. A page with none of them belongs to something else.
CONTINUATION_WORDS = ["AMOUNT HAULED", "WRITTEN AGREEMENT", "CERTIFICATION", "SIGNATURE",
                      "METHOD USED", "PROCESS WASTEWATER", "CONTINUED", "D-2", "G-2", "F-2"]

PARAMETERS_DF = pd.read_csv(DATA_ROOT / "parameters.csv")
LOCATIONS_DF = pd.read_csv(DATA_ROOT / "parameter_locations.csv")
TEMPLATES_DF = pd.read_csv(DATA_ROOT / "templates.csv")

# parameter_key maps to column name and data type, and column name maps to data type
COLS = dict(zip(PARAMETERS_DF["parameter_key"], PARAMETERS_DF["parameter_name"]))
PARAM_TYPES = dict(zip(PARAMETERS_DF["parameter_key"], PARAMETERS_DF["data_type"]))
COL_TYPES = dict(zip(PARAMETERS_DF["parameter_name"], PARAMETERS_DF["data_type"]))

# Each form template's fields are listed in parameter_locations.csv
UNKNOWN_TEMPLATE = "unknown" # assumed to be the R5 2007 general order
PARAM_TEMPLATE = dict(zip(TEMPLATES_DF["template_key"], TEMPLATES_DF["parameter_template"]))
PARAM_TEMPLATE[UNKNOWN_TEMPLATE] = "R5-2007-0035_general_order"

# common_errors.csv lists OCR misreadings to fix, and "cs" marks the case-sensitive ones.
# TODO: dropna discards the "; " row, whose correction is blank, so it is never applied.
errors = pd.read_csv(DATA_ROOT / "common_errors.csv").dropna(subset=["error", "correct"])
CI_ERRORS = dict(errors.loc[~errors["cs"], ["error", "correct"]].values)
CS_ERRORS = dict(errors.loc[errors["cs"], ["error", "correct"]].values)
STRIP_CHARS = [",", "=", ":", "___"]

# Figures (ColorBrewer palette)
COLORS = {"manure": "#8c510a", "wastewater": "#35978f", "manual": "#01665e",
          "extracted": "#bf812d", "accuracy": "#1f78b4"}
plt.rcParams["font.size"] = 14

In [ ]:

# Regex patterns for load extraction
# Drafted and adjusted from https://techearl.com/regex-cheat-sheet and Claude

NUM = r"(\d+(?:\.\d+)?)"
LOAD = r"(?:(?:truck|tanker\s*)?loa[dp]s?|hauls?|dump\s*t(?:k|ruck)s?)"
UNIT = r"(tons?|gallons?|gals?|yards?)"  # gallon before gal: callers test "gallon" in unit
SEP = r"\s*(?:[x×@\-]|at)\s*"

LOAD_PATTERNS = [
    (re.compile(rf"{NUM}\s*{LOAD}(?:\s+\w+)*?{SEP}(?:approx\.?\s*)?{NUM}\s*{UNIT}", re.I), (0, 1, 2)),
    (re.compile(rf"{NUM}\s*{UNIT}{SEP}{NUM}\s*{LOAD}", re.I), (2, 0, 1)),
    (re.compile(rf"{NUM}\s*{LOAD}\s+{NUM}\s*{UNIT}\s+each\b", re.I), (0, 1, 2)),
]

HOURS_RE = re.compile(r"(\d+(?:\s*\d+/\d+)?(?:\.\d+)?)\s*(?:hours?|hrs?)\b", re.I)
GPM_RE = re.compile(r"(\d+(?:,\d+)?(?:\.\d+)?)\s*(?:gpm|gallons per min)\b", re.I)
TABLE_ROW_RE = re.compile(rf"^(.+?)\s+([\d,]+)\s+{UNIT}\s+(\d+)\s*%", re.I)
PHONE_RE = re.compile(r"\(?\d{3}\)?[\s\-\.]?\d{3,4}[\s\-\.]?\d{4}")

# Haul dates
MONTHS = "|".join(calendar.month_name[1:] + calendar.month_abbr[1:]).lower()
MONTH_NUM = {m.lower(): i for names in (calendar.month_name, calendar.month_abbr)
             for i, m in enumerate(names) if m}
DATE_TOKEN_RE = re.compile(
    rf"\d{{1,2}}/\d{{1,2}}/\d{{2,4}}"
    rf"|\d{{1,2}}-\d{{1,2}}-\d{{2,4}}"
    rf"|\d{{1,2}}\.\d{{1,2}}\.\d{{2,4}}"
    rf"|(?:{MONTHS})(?:\s+\d{{1,2}})?(?:\s*,?\s*\d{{2,4}})?",
    re.I,
)
MONTH_SPAN_RE = re.compile(rf"\b({MONTHS})\.?\s*[-–—]\s*({MONTHS})\.?", re.I)
MONTH_DAYS_RE = re.compile(
    rf"\b({MONTHS})\.?\s+(\d{{1,2}})\s*(?:[-–—]|thru|through|to|&|,|\s)\s*(\d{{1,2}})\b(?!\d)", re.I
)
SHARED_MONTH_RE = re.compile(
    r"(?<![\d/])(\d{1,2})/(\d{1,2})\s*[-–—]\s*(\d{1,2})(?:/((?:19|20)\d{2}|24))?(?![\d/])"
)
YEAR_RE = re.compile(r"\b(19|20)\d{2}\b")
DAY_RE = re.compile(r"(?<!\d)\d{1,2}(?!\d)")

# Look for whitespace before parcel number
PARCEL_RE = re.compile(r"(?:\(?\d*\)?\s*[Xx]?\s+)?([\dXx]{2,}\s*[.\-]\s*[\dXx]{2,}(?:\s*[.\-]\s*[\dXx]+)*)")

KEEP_UPPER = {"LLC", "GPM", "INC", "CA", "DBA", "NA", "N/A"}

OCR and page detection helpers

In [ ]:
def ocr_txt_paths():
    """Find each PDF's OCR text file. Use LLMWhisperer's text when there is one, else tesseract's."""
    paths = {}
    for method in ["tesseract", "llmwhisperer"]:  # the later method overwrites
        for path in sorted((DATA_ROOT / REGION).glob(f"*/*/{method}_output/*.txt")):
            paths[path.stem] = path
    return paths


def ocr_output_path(pdf_path, method):
    """Return where a method's text for a PDF is saved: data/R5/<county>/<template>/<method>_output/<name>.txt"""
    pdf_path = Path(pdf_path)
    return pdf_path.parents[1] / f"{method}_output" / f"{pdf_path.stem}.txt"


def save_text(path, text):
    """Write text to a file, creating its folder if needed."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def join_pages(texts):
    """Join {page number: text} into one text, with a "=== Page N ===" line before each page."""
    return "\n\n".join(f"=== Page {page} ===\n{text}" for page, text in texts.items())


def rotate_upright(image):
    """Turn a page image upright, using Tesseract's guess of which way is up."""
    try:
        angle = int(pytesseract.image_to_osd(image, output_type=Output.DICT)["rotate"])
    except pytesseract.TesseractError:  # a nearly blank page has no orientation
        return image
    # Tesseract's angle is always 0, 90, 180 or 270 degrees clockwise
    return np.ascontiguousarray(np.rot90(image, -angle // 90))


def detect_manifest_pages_in_pdf(pdf_path):
    """Find the pages that mention a manifest, plus the page after each one. The search starts
    at the report's "Required Attachments" list, and scanned pages are read with OCR."""
    with fitz.open(pdf_path) as doc:
        texts = [page.get_text().lower() for page in doc]
        start = next((i for i, text in enumerate(texts) if "required attachments" in text), 0)
        found = set()
        for i in range(start, len(texts)):
            text = texts[i]
            if len(text.strip()) < 50:  # a scanned page: OCR the top two-thirds of a low-resolution image
                pix = doc[i].get_pixmap(dpi=120)
                image = np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width, pix.n)
                text = pytesseract.image_to_string(rotate_upright(image[: pix.height // 3 * 2]),
                                                   config="--psm 6 --oem 1").lower()
            if any(term in text for term in MANIFEST_TERMS):
                found |= {i + 1, i + 2}
    return sorted(page for page in found if page <= len(texts))


def fitz_text(pdf_path, pages):
    """Get the PDF's own embedded text for each page that has any."""
    with fitz.open(pdf_path) as doc:
        texts = {page: doc[page - 1].get_text() for page in pages}
    return join_pages({page: text for page, text in texts.items() if text.strip()})


def tesseract_text(pdf_path, pages):
    """OCR each page with tesseract, after turning it upright and boosting its contrast.
    Return the text and the mean word confidence."""
    config = "--oem 1 --psm 4 -c preserve_interword_spaces=1"
    texts, confidences = {}, []
    for page in pages:
        # A few scans are too large for PIL at 350 DPI, but 250 always fits
        try:
            image = convert_from_path(pdf_path, dpi=350, first_page=page, last_page=page)[0]
        except Image.DecompressionBombError:
            image = convert_from_path(pdf_path, dpi=250, first_page=page, last_page=page)[0]
        gray = cv2.cvtColor(rotate_upright(np.array(image)), cv2.COLOR_RGB2GRAY)
        gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
        _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        texts[page] = pytesseract.image_to_string(thresh, config=config)
        words = pytesseract.image_to_data(thresh, config=config, output_type=Output.DICT)
        confidences += [c for c in words["conf"] if c != -1]
    return join_pages(texts), (sum(confidences) / len(confidences) if confidences else 0)


def llmwhisperer_text(pdf_path, pages):
    """OCR the pages with the LLMWhisperer API, which can read handwriting that tesseract cannot."""
    with open(pdf_path, "rb") as f:
        response = requests.post(
            f"{LLMWHISPERER_BASE_URL}/whisper",
            headers={**LLMWHISPERER_HEADERS, "Content-Type": "application/pdf"},
            params={"mode": "form", "timeout": 300, "output_mode": "line-printer",
                    "page_separator": PAGE_BREAK, "force_text_processing": "true",
                    "pages_to_extract": ",".join(map(str, pages))},
            data=f.read(),
            timeout=600,
        )
    if response.status_code != 202:
        raise RuntimeError(f"LLMWhisperer error {response.status_code}: {response.text[:500]}")
    job = {"whisper_hash": response.json()["whisper_hash"]}

    for _ in range(300):
        time.sleep(2)
        status = requests.get(f"{LLMWHISPERER_BASE_URL}/whisper-status",
                              headers=LLMWHISPERER_HEADERS, params=job, timeout=30)
        status.raise_for_status()
        state = status.json()["status"]
        if state == "processed":
            result = requests.get(f"{LLMWHISPERER_BASE_URL}/whisper-retrieve",
                                  headers=LLMWHISPERER_HEADERS, params=job, timeout=60)
            result.raise_for_status()
            chunks = [chunk.strip() for chunk in result.json()["result_text"].split(PAGE_BREAK)]
            return join_pages({page: chunk for page, chunk in zip(pages, chunks) if chunk})
        if state not in ("processing", "accepted"):
            raise RuntimeError(f"LLMWhisperer error status: {state}")
    raise TimeoutError("Timeout waiting for LLMWhisperer results")


Helpers for splitting OCR text into manifests and extracting their fields

In [ ]:
def coerce_columns(df):
    """Convert each column to its data type from parameters.csv. Changes df in place and returns it."""
    for col in df.columns:
        dtype = COL_TYPES.get(col)
        if dtype == "numeric":
            df[col] = pd.to_numeric(df[col], errors="coerce")
        elif dtype == "date":
            df[col] = pd.to_datetime(df[col], format="mixed", errors="coerce")
        elif dtype == "boolean":
            df[col] = df[col].astype(str).str.strip().str.upper().map({"TRUE": True, "FALSE": False, "NAN": None})
    return df


def split_pages(ocr_text):
    """Split OCR text into {page number: page text} at each "=== Page N ===" line."""
    markers = list(PAGE_RE.finditer(ocr_text))
    return {int(m.group(1)): ocr_text[m.end(): nxt.start() if nxt else None].strip()
            for m, nxt in zip(markers, markers[1:] + [None])}


def clean_common_errors(text):
    """Fix common OCR misreadings and strip noise characters, dropping blank lines."""
    for old, new in CI_ERRORS.items():
        text = re.sub(re.escape(old), new, text, flags=re.IGNORECASE)
    for old, new in CS_ERRORS.items():
        text = text.replace(old, new)
    for char in STRIP_CHARS:
        text = text.replace(char, "")
    text = re.sub(r"(\d)[lI](\d)", r"\1l\2", text)
    text = re.sub(r"([a-zA-Z])[0O]([a-zA-Z])", r"\1O\2", text)
    text = re.sub(r"\s*([/-])\s*", r"\1", text)
    text = text.replace("_", "")  # after the line above, so "OL\n_-r" stays two lines
    text = re.sub(r"[ \t\f\v\r]+", " ", text)
    return "\n".join(re.sub(r"^\s*;\s*", "", line) for line in text.split("\n") if line.strip())


def looks_like_manifest_first_page(page_upper):
    """Check whether a page is a manifest's first page. It must name the manifest form, a tracking
    or attachment heading, and a heading that only first pages have."""
    return (
        "MANIFEST" in page_upper
        and any(word in page_upper for word in ["TRACKING", "ATTACHMENT"])
        and any(word in page_upper for word in [
            "INSTRUCTIONS", "COMPLETE ONE", "WASTE GENERATOR INFORMATION", "ADDRESS OF HAULING",
        ])
    )


def find_template(text_upper, page_count):
    """Find the first templates.csv row that matches the manifest text. Specific forms must sit
    above generic ones in the file."""
    # Keywords "A|B && !C" mean A or B must appear and C must not. A blank page_count matches any length.
    for _, row in TEMPLATES_DF.iterrows():
        if pd.notna(row["page_count"]) and row["page_count"] != page_count:
            continue
        groups = [group.strip() for group in row["keywords"].upper().split("&&")]
        if all(
            not any(word.strip() in text_upper for word in group[1:].split("|"))
            if group.startswith("!")
            else any(word.strip() in text_upper for word in group.split("|"))
            for group in groups
        ):
            return row["template_key"]
    return UNKNOWN_TEMPLATE


def identify_manifest_pages(ocr_text):
    """Split a report's OCR text into manifests. Return each one's text, (first page, last page)
    and form template."""
    pages = split_pages(ocr_text)
    page_numbers = sorted(pages)
    manifests, last_used = [], 0
    for idx, first_page in enumerate(page_numbers):
        first_upper = pages[first_page].upper()
        if (first_page <= last_used or "REQUIRED ATTACHMENTS" in first_upper
                or not looks_like_manifest_first_page(first_upper)):
            continue
        combined, last_page = pages[first_page], first_page

        # Most manifests run onto a second page, unless page 1 already has the signature. Take the
        # next page only if it continues the form, since a lab report can follow a one-page form.
        if (idx + 1 < len(page_numbers) and "CERTIFICATION" not in first_upper
                and "SIGNATURE OF HAULER" not in first_upper):
            second_upper = pages[page_numbers[idx + 1]].upper()
            starts_another = ("REQUIRED ATTACHMENTS" not in second_upper
                              and looks_like_manifest_first_page(second_upper))
            if not starts_another and any(word in second_upper for word in CONTINUATION_WORDS):
                last_page = page_numbers[idx + 1]
                combined += "\n\n" + pages[last_page]

        # A "page 2 of 3" form has a third page with no manifest keywords of its own
        if "PAGE 2 OF 3" in combined.upper() and last_page + 1 in pages:
            last_page += 1
            combined += "\n" + pages[last_page]

        last_used = last_page
        cleaned = clean_common_errors(combined)
        template = find_template(cleaned.upper(), last_page - first_page + 1)
        manifests.append((cleaned, (first_page, last_page), template))
    return manifests


def parse_hauling_table(manifest_text):
    """Read the rows of the hauling table on one-page "Solid/Liquid Manure" forms."""
    lines = manifest_text.split("\n")
    header = next((i for i, line in enumerate(lines) if "date" in line.lower() and "haul" in line.lower()), None)
    if header is None:
        return []
    rows = []
    for line in lines[header + 1:]:
        line = line.strip()
        if not line or line.lower().startswith("total"):
            break
        match = TABLE_ROW_RE.match(line)
        if match:
            date_range, amount, units, moisture = match.groups()
            # Tons and yards are manure, and gallons are wastewater
            is_manure = "ton" in units.lower() or "yard" in units.lower()
            amount_key = "manure_amount" if is_manure else "wastewater_amount"
            rows.append({COLS["haul_date"]: date_range.strip(), COLS[amount_key]: amount.replace(",", ""),
                         COLS["manure_moisture_percent"]: moisture})
    return rows


def parse_compact_date(text):
    """Parse compact numeric dates where OCR drops or misreads the '/' separator."""
    if len(text) not in (3, 4) or not text.isdecimal():
        return None
    if len(text) == 3:
        month, day = int(text[0]), int(text[1:])
    elif text[0] != "0" and text[1] == "1":  # OCR read the "/" as a "1"
        month, day = int(text[0]), int(text[2:])
    else:
        month, day = int(text[:2]), int(text[2:])
    try:
        return datetime(YEAR, month, day)
    except ValueError:
        return None


def make_date(month, day, year=None):
    """Build a date, or NaT if that month/day does not exist.
    A missing year, or a bare "24", means the report year (not the year 24 AD)."""
    if year in (None, "24"):
        year = YEAR
    return pd.to_datetime(f"{month}/{day}/{year}", format="%m/%d/%Y", errors="coerce")


def shorthand_dates(text):
    """Return the endpoints of haul-date range shorthand: "7/2-3/24", "Sep 24-25 2024",
    "June-Oct 2024", "Entire Year". DATE_TOKEN_RE matches none of these on its own."""
    if re.search(r"ent[il]r?[ve]e?\s*year|all\s*year|year\s*round", text, re.I):
        return [make_date(1, 1), make_date(12, 31)]

    found = []
    for match in MONTH_SPAN_RE.finditer(text):  # "June-Oct 2024"
        first, last = MONTH_NUM[match.group(1).lower()], MONTH_NUM[match.group(2).lower()]
        found += [make_date(first, 1), make_date(last, 31 if last == 12 else 1)]

    for match in MONTH_DAYS_RE.finditer(text):  # "September 24-25 2024"
        month = MONTH_NUM[match.group(1).lower()]
        found += [make_date(month, int(match.group(2))), make_date(month, int(match.group(3)))]

    for match in SHARED_MONTH_RE.finditer(text):  # "7/2-3/24"
        month, day1, day2 = int(match.group(1)), int(match.group(2)), int(match.group(3))
        year = match.group(4)
        # "6/1-7/24" usually means 6/1 through 7/24 rather than two days in month 6.
        # Prefer that reading when "day2/24" is itself a real, later date.
        second_is_later = day2 > month or (day2 == month and day1 < 24)
        if year == "24" and second_is_later:
            found += [make_date(month, day1), make_date(day2, 24)]
        else:
            found += [make_date(month, day1, year), make_date(month, day2, year)]

    return [date for date in found if pd.notna(date)]


def split_haul_dates(data):
    """Parse haul_date into first and last dates. Leaves fields blank rather than defaulting."""
    haul_date = data[COLS["haul_date"]]
    if not haul_date:
        return

    parsed = shorthand_dates(haul_date)

    # Fall back to splitting on separators only when nothing matched, else "7/2-3/24"
    # would also yield a spurious 3/24.
    date_parts = DATE_TOKEN_RE.findall(haul_date)
    if not date_parts and not parsed:
        date_parts = [part.strip() for part in
                      re.split(r"[-–—]|\bto\b|,|;|&", haul_date, flags=re.I) if part.strip()]

    for part in date_parts:
        compact = parse_compact_date(part)
        if compact is not None:
            parsed.append(compact)
            continue

        part = re.sub(r"(\d{1,2})((?:19|20)\d{2})", r"\1 \2", part)

        # A bare month name means the whole month: its first day, or 12/31 for December
        month_number = MONTH_NUM.get(part.lower())
        if month_number:
            parsed.append(datetime(YEAR, month_number, 31 if month_number == 12 else 1))
            continue

        if not DAY_RE.search(part):
            continue
        try:
            parsed_date = date_parser.parse(part, default=datetime(YEAR, 1, 1))
        except date_parser.ParserError:
            # Free text such as "as needed" is not a date, but the other parts still count
            continue
        parsed.append(parsed_date if YEAR_RE.search(part) else parsed_date.replace(year=YEAR))

    if parsed:
        parsed.sort()
        first, last = parsed[0], parsed[-1]
        # A single date is recorded as the last date only, leaving the first date blank
        data[COLS["haul_date_first"]] = f"{first.month}/{first.day}/{first.year}" if len(parsed) > 1 else None
        data[COLS["haul_date_last"]] = f"{last.month}/{last.day}/{last.year}"


def text_without_match(text, match):
    """The text with one regex match cut out and the two sides rejoined by a space."""
    before, after = text[: match.start()].strip(), text[match.end():].strip()
    return " ".join(part for part in [before, after] if part)


def parse_destination_address_and_parcel(text):
    """Split a destination like "1234 W Main Ave 022-035-012" into (address, parcel number)."""
    matches = list(PARCEL_RE.finditer(text))
    if not matches:
        return text, None
    parcel = "".join(matches[-1].group(1).split())
    rest = text_without_match(text, matches[-1])
    # What is left of the line is only an address if it has letters and some length
    address = rest if len(rest) >= 5 and any(char.isalpha() for char in rest) else None
    return address, parcel


def normalize_apn(parcel_number):
    """Write a parcel number as digits joined by hyphens, reading an OCR'd "X" as a zero.
    Return None when the text holds no parcel number."""
    match = re.match(r"[\d\s.\-Xx]+", parcel_number.strip())
    if not match:
        return None
    return "".join(match.group().split()).upper().replace(".", "-").replace("X", "0")


def split_apn_county(parcel_text):
    """Split a value like "022-035-012 Fresno" into (normalized APN, county name)."""
    match = re.match(r"([\d\s.\-Xx]+)\s+([A-Za-z].*)$", parcel_text.strip())
    if not match:
        return normalize_apn(parcel_text), None
    return normalize_apn(match.group(1)), match.group(2).strip()


def smart_title(text):
    """Title-case a value, keeping abbreviations such as LLC in caps and not capitalizing
    after an apostrophe (str.title() alone turns "farmer's" into "Farmer'S")."""
    words = []
    for word in text.split():
        if word.upper().strip(".,;:()") in KEEP_UPPER:
            words.append(word.upper())
            continue
        head, *rest = word.title().split("'")
        words.append("'".join([head] + [part[:1].lower() + part[1:] for part in rest]))
    return " ".join(words)


def extract_value_from_line(line, item_order, ignore_before, ignore_after):
    """Get the value from one OCR line. Take the text after ignore_before, cut it at the first
    ignore_after marker, then pick the item_order-th word if one is given."""
    if pd.notna(ignore_before):
        start = line.lower().find(ignore_before.lower())
        if start != -1:
            line = line[start + len(ignore_before):]
    cut = len(line)
    for marker in ignore_after:
        if marker == "first_number":
            # The next field on the form begins at its own number (a zip, a phone)
            number = re.search(r"\(?\d{2,}", line)
            found = number.start() if number else -1
        else:
            found = line.lower().find(marker.lower())
        if found != -1:
            cut = min(cut, found)
    line = line[:cut].strip()
    if pd.notna(item_order):
        words = line.split()
        return words[int(item_order)] if int(item_order) < len(words) else ""
    return line


def find_parameter_value(ocr_text, row):
    """Find one parameter's value in the manifest text, using one row of parameter_locations.csv."""
    data_type = PARAM_TYPES[row["parameter_key"]]
    default = np.nan if data_type == "numeric" else None

    search_text = ocr_text
    heading = row["page_search_text"]
    if pd.notna(heading):
        # Some labels appear twice on a form, so only search after this section heading
        heading_pos = ocr_text.lower().find(heading.lower())
        if heading_pos == -1:
            return default
        search_text = ocr_text[heading_pos + len(heading):]

    label = row["row_search_text"].lower()
    direction = row["search_direction"]
    ignore_after = [] if pd.isna(row["ignore_after"]) else [m.strip() for m in row["ignore_after"].split("|")]
    def value_on(text):
        return extract_value_from_line(text, row["item_order"], row["ignore_before"], ignore_after)

    lines = [line.strip() for line in search_text.split("\n") if line.strip()]
    label_idx = next((i for i, line in enumerate(lines) if label in line.lower()), None)
    if label_idx is None:
        return default
    line = lines[label_idx]
    next_line = lines[label_idx + 1] if label_idx + 1 < len(lines) else None

    value = None
    if direction in ("right", "right_below"):
        right_text = line[line.lower().find(label) + len(label):].strip()
        value = value_on(right_text) if right_text else None
        section_starts = ("enter the amount", "process wastewater", "written agreement", "method used")
        if direction == "right_below" and next_line and not next_line.lower().startswith(section_starts):
            value = f"{value} {value_on(next_line)}" if value else value_on(next_line)
    elif direction == "above" and label_idx > 0:
        value = value_on(lines[label_idx - 1])
    elif direction == "below" and next_line:
        if pd.notna(row["ignore_before"]) and next_line.lower().startswith(row["ignore_before"].lower()):
            return default
        value = value_on(next_line)

    if not value or not value.strip():
        return default
    if data_type == "numeric":
        try:
            value = float(value.replace(",", ""))
        except ValueError:
            pass
    if (data_type == "numeric" and (pd.isna(value) or value == 0)) or value in ("N/A", "NA", "."):
        return default
    return smart_title(value) if isinstance(value, str) else value


def clean_destination_fields(data):
    """Separate parcel numbers and phone numbers from the destination address fields, and
    drop destination types that just repeat the form's own wording."""
    address = data[COLS["destination_address"]]
    if address:
        address_part, parcel_part = parse_destination_address_and_parcel(address)
        if parcel_part and not data[COLS["destination_parcel_number"]]:
            data[COLS["destination_parcel_number"]] = parcel_part
        if address_part or parcel_part:
            # Only a parcel number means no address was written on the form
            data[COLS["destination_address"]] = address_part

    contact = data[COLS["destination_contact_address"]]
    if contact:
        phones = list(PHONE_RE.finditer(contact))
        if phones:  # drop the last phone number, keeping the address around it
            contact = text_without_match(contact, phones[-1]) or None
        street_number = re.search(r"\b\d{2,}", contact) if contact else None
        if street_number:
            contact = contact[street_number.start():].strip()
        data[COLS["destination_contact_address"]] = contact

    destination_type = data[COLS["destination_type"]]
    if destination_type and ("(as identified" in destination_type.lower() or "above)" in destination_type.lower()):
        data[COLS["destination_type"]] = None

    parcel = data[COLS["destination_parcel_number"]]
    # Split a value like "022-035-012 Kings" into its parcel number and county
    data[COLS["destination_parcel_number"]], data[COLS["destination_county"]] = (
        split_apn_county(parcel) if parcel else (None, None))


def parse_method_text(data):
    """Read hauls, amount per haul, hours pumped and pumping rate from the "method used" text."""
    for waste_type, units in [("manure", ["ton", "yard"]), ("wastewater", ["gallon"])]:
        method_text = data[COLS[f"{waste_type}_method"]]
        if not method_text:
            continue
        for regex, (load_idx, amount_idx, unit_idx) in LOAD_PATTERNS:
            match = regex.search(method_text)
            if match:
                groups = match.groups()
                data[COLS[f"{waste_type}_number_hauls"]] = groups[load_idx]
                for unit in units:
                    if unit in groups[unit_idx].lower():
                        data[COLS[f"{waste_type}_{unit}_per_haul"]] = groups[amount_idx]
                break

    method_text = data[COLS["wastewater_method"]]
    if method_text:
        hours_match = HOURS_RE.search(method_text)
        if hours_match:
            hours = hours_match.group(1).replace(" ", "")
            fraction = re.match(r"(\d*?)(\d)/(\d+)", hours)
            if fraction:  # "1 1/2 hours" -> 1.5
                whole, numerator, denominator = fraction.groups()
                hours = float(whole or 0) + float(numerator) / float(denominator)
            data[COLS["wastewater_hours_pumped"]] = hours
        gpm_match = GPM_RE.search(method_text)
        if gpm_match:
            data[COLS["wastewater_pumping_rate"]] = gpm_match.group(1)


def classify_transport(data, manifest_text):
    """Fix solids reported as moisture, and mark pipeline and trucked transport."""
    # The form has one box for both, so a value followed by "% moisture" is not solids
    solids, moisture = data[COLS["manure_solids_percent"]], data[COLS["manure_moisture_percent"]]
    if solids and not moisture:
        solids_text = f"{solids:g}" if isinstance(solids, float) else str(solids)
        if re.search(rf"{re.escape(solids_text)}\s*%\s*moisture", manifest_text, re.I):
            data[COLS["manure_moisture_percent"]], data[COLS["manure_solids_percent"]] = solids, None

    is_pipeline = "pipeline" in manifest_text.lower()
    data[COLS["is_pipeline"]] = is_pipeline
    method = (data[COLS["wastewater_method"]] or "").lower()
    if is_pipeline:
        data[COLS["is_trucked"]] = False
    elif (any(word in method for word in ["load", "haul", "tank"])
          and not any(word in method for word in ["apply", "applied"])):
        data[COLS["is_trucked"]] = True
    else:
        data[COLS["is_trucked"]] = "UNSURE"


def extract_manifest_fields(manifest_text, template):
    """All parameter values for one manifest, using the template's field locations."""
    data = dict.fromkeys(COLS.values())
    for _, row in LOCATIONS_DF[LOCATIONS_DF["template"] == template].iterrows():
        data[COLS[row["parameter_key"]]] = find_parameter_value(manifest_text, row)
    clean_destination_fields(data)
    parse_method_text(data)
    classify_transport(data, manifest_text)
    return data


def extract_manifests_from_txt(txt_path):
    """Extract every manifest record from one report's OCR text, and save each manifest's text to
    all_manifests/<pdf name>/manifest_N.txt. A one-page hauling table gives one record per row."""
    manifests = []
    out_dir = DATA_ROOT / REGION / txt_path.parents[2].name / "all_manifests" / txt_path.stem
    for i, (text, (first_page, last_page), form_template) in enumerate(
            identify_manifest_pages(txt_path.read_text(encoding="utf-8")), start=1):
        parameter_template = PARAM_TEMPLATE[form_template]
        record = {**extract_manifest_fields(text, parameter_template),
                  "Parameter Template": parameter_template, "Form Template": form_template,
                  "Source PDF": txt_path.stem, "Start Page": first_page, "End Page": last_page,
                  "County": txt_path.parents[2].name}
        table_rows = parse_hauling_table(text) if parameter_template == "R5-2007-0035_one_page_2" else []
        if table_rows:
            records = [{**record, **row, "Manifest Number": f"{i}{string.ascii_lowercase[j]}"}
                       for j, row in enumerate(table_rows)]
        else:
            records = [{**record, "Manifest Number": i}]
        for record in records:
            split_haul_dates(record)
        manifests += records
        save_text(out_dir / f"manifest_{i}.txt", text)
    return manifests


def save_manifest_pdfs(txt_path):
    """Split one report's manifests into all_manifests/<stem>/manifest_N.pdf plus an
    all_manifests.pdf with every manifest page. Rewrites ~1,600 PDFs when run for all reports."""
    pdf_path = next(p for p in (txt_path.parents[1] / "original").iterdir() if p.stem == txt_path.stem)
    out_dir = DATA_ROOT / REGION / txt_path.parents[2].name / "all_manifests" / txt_path.stem
    with fitz.open(pdf_path) as doc, fitz.open() as all_pages:
        for i, (_, (first_page, last_page), _) in enumerate(
                identify_manifest_pages(txt_path.read_text(encoding="utf-8")), start=1):
            with fitz.open() as manifest:
                manifest.insert_pdf(doc, from_page=first_page - 1, to_page=last_page - 1)
                all_pages.insert_pdf(doc, from_page=first_page - 1, to_page=last_page - 1)
                manifest.save(out_dir / f"manifest_{i}.pdf")
        all_pages.save(out_dir / "all_manifests.pdf")

Post-processing and geocoding helpers

In [ ]:
SPECIFIC_COLS = {
    mtype: set(PARAMETERS_DF.loc[PARAMETERS_DF["manifest_type"] == mtype, "parameter_name"])
    for mtype in ["wastewater", "manure"]
}
METADATA_COLS = ["Source PDF", "Manifest Number", "Start Page", "End Page"]

# Haul-size bins, {label: (lower, upper, truck capacity)}. Manure loads of 5-15 tons count as
# a 10-ton truck, and 15-25 tons as a 20-ton truck.
WASTE_TYPES = {
    "manure": dict(rate="manure_ton_per_haul", hauls="manure_number_hauls", amount="manure_amount",
                   bins={"10-ton": (5, 15, 10.0), "20-ton": (15, 25, 20.0)}),
    "wastewater": dict(rate="wastewater_gallon_per_haul", hauls="wastewater_number_hauls",
                       amount="wastewater_amount",
                       bins={"<8,000 gal": (-np.inf, 8000, 2000.0), ">=8,000 gal": (8000, np.inf, 10000.0)}),
}

DEST_TYPE_MAP = {
    "Broker": ["broker"],
    "Composting Facility": ["compost", "kellogg", "hyponex", "fertilizer", "supply"],
    "Farmer": ["farm"],
}

COORD_RE = re.compile(r"\s*\(?\s*(-?\d+\.\d+)\s*,\s*(-?\d+\.\d+)\s*\)?\s*$")

# Data folders that are not counties, and the county to add to their address searches
# (None lets the address's own city or ZIP decide). The other folders are counties.
FOLDER_COUNTIES = {"tulare_east": "Tulare", "tulare_west": "Tulare", "fresno_madera": None, "rancho_cordova": None}

CA_EXTENT = "-124.55,32.20,-114.10,42.10"  # xmin,ymin,xmax,ymax, pins ArcGIS to CA
ARCGIS_URL = "https://geocode.arcgis.com/arcgis/rest/services/World/GeocodeServer/findAddressCandidates"
PARCEL_URL = ("https://gis.water.ca.gov/arcgis/rest/services/Location/Geocoding_Parcels_APN_TaxAPN/"
              "GeocodeServer/findAddressCandidates")
ROAD_TYPES = {"ave", "avenue", "st", "street", "rd", "road", "blvd",
              "ln", "lane", "dr", "drive", "way", "hwy", "highway"}

# The counties each ZIP code covers. A ZIP near a county line can cover two ("95380" is Stanislaus and Merced).
ZIP_COUNTIES = (pd.read_csv(DATA_ROOT / "zipcode_to_county.csv", usecols=["zip", "county_names_all"], dtype=str)
                .set_index("zip")["county_names_all"].str.split("|").to_dict())
COUNTY_NAMES = {county for counties in ZIP_COUNTIES.values() for county in counties}
# A county name at the end of an address field ("West & American Fresno"). It is built from real
# county names, so "Baker Road Stanislaus County" cannot read as a county named "Baker Road Stanislaus".
COUNTY_SUFFIX_RE = re.compile(
    r"\b(" + "|".join(re.escape(c) for c in COUNTY_NAMES) + r")(?:\s+County)?\s*$", re.I
)

CACHE_PATH = Path("output_data/geocode_cache.json")
cache = json.loads(CACHE_PATH.read_text())


def cache_key(*parts):
    return "|".join(str(part) for part in parts if part)


def cache_put(key, value):
    """Store a geocoding result and rewrite the cache file, so an interrupted run loses nothing."""
    cache[key] = value
    CACHE_PATH.write_text(json.dumps(cache, indent=2))
    return value


def arcgis_geocoder(query):
    """Look up an address with ArcGIS, limited to California. Return (address, lat, lng, address type), or None."""
    time.sleep(0.2)
    response = requests.get(ARCGIS_URL, timeout=15, params={
        "SingleLine": query, "f": "json", "maxLocations": 1,
        "countryCode": "USA", "searchExtent": CA_EXTENT, "outFields": "Addr_type",
    })
    response.raise_for_status()
    candidates = response.json()["candidates"]
    if not candidates:
        return None
    top = candidates[0]
    return top["address"], top["location"]["y"], top["location"]["x"], top["attributes"]["Addr_type"]


# Google is a fallback for addresses ArcGIS cannot place at street level
GOOGLE_API_KEY = os.environ.get("GOOGLE_GEOCODING_API_KEY")
google_geocoder = GOOGLE_API_KEY and RateLimiter(
    GoogleV3(api_key=GOOGLE_API_KEY).geocode, min_delay_seconds=0.1, error_wait_seconds=1.0,
    swallow_exceptions=False)


def google_lookup(query):
    """Look up an address with Google, if there is an API key. Return (address, lat, lng), or None."""
    hit = google_geocoder and google_geocoder(query, components={"country": "US", "administrative_area": "CA"})
    return (hit.address, hit.latitude, hit.longitude) if hit else None


def format_intersection(text, county_name=None):
    """"Muscat & Westlawn" -> "Muscat & Westlawn, Fresno County, CA", else None.

    Only a crossing of two streets is a point.
    Assume anything with a digit already has a house number.
    """
    cleaned = " ".join(text.split())
    if any(char.isdigit() for char in cleaned):
        return None
    county_suffix = COUNTY_SUFFIX_RE.search(cleaned)
    if county_suffix:
        county_name = county_suffix.group(1).title()
        cleaned = cleaned[: county_suffix.start()].strip()
    # Treat "A and B" the same as "A & B", then there must be exactly two sides
    sides = " ".join("&" if word.lower() == "and" else word for word in cleaned.split()).split("&")
    if len(sides) != 2:
        return None
    left_words, right_words = sides[0].split(), sides[1].split()
    if not (1 <= len(left_words) <= 2 and 1 <= len(right_words) <= 2):
        return None
    # Take the word next to the "&", since a farmer name can lead ("Bottasso Muscat & Westlawn"),
    # but step back over a trailing street type ("Baker Road & ...")
    first = (left_words[0] if left_words[-1].lower() in ROAD_TYPES else left_words[-1]).strip(".,'")
    second = right_words[0].strip(".,'")
    if not (first.isalpha() and second.isalpha()):
        return None
    return f"{first} & {second}, {county_name} County, CA" if county_name else f"{first} & {second}, CA"


def parcel_candidates(apn):
    """Find every parcel the DWR locator lists under a parcel number, as [lat, lng, county, address].
    Each county numbers its own parcels, so one number can match parcels in several counties."""
    key = cache_key("parcels", apn)
    if key in cache:
        return cache[key]
    # DWR knows parcels by several spellings, so list each reading of the number's digits
    digits = apn.replace("-", "")
    if len(digits) == 16:  # book, page, parcel and suffix in 4 digits each, like "0045-0190-0044-0000"
        readings = [digits[1:4] + digits[5:8] + digits[9:12]]
        if digits[4:6] == "00":  # some forms write page 190 as "0019"
            readings.append(digits[1:4] + digits[6:8] + "0" + digits[9:12])
    elif len(digits) == 8:  # San Joaquin's 3-3-2 numbers, which DWR also stores as 3-3-3 ("185-100-030")
        readings = [digits, digits + "0"]
    else:
        readings = [digits]
    spellings = [apn]
    for reading in readings:
        spellings.append(reading)
        if len(reading) == 9:
            dashed = f"{reading[:3]}-{reading[3:6]}-{reading[6:]}"
            spellings += [dashed, f"{dashed}-000"]  # some counties add a "-000" suffix
    found = []
    for spelling in dict.fromkeys(spellings):
        response = requests.get(PARCEL_URL, timeout=60, params={
            "SingleLine": spelling, "f": "json", "outFields": "Subregion", "maxLocations": 20})
        response.raise_for_status()
        for candidate in response.json()["candidates"]:
            parcel = [round(candidate["location"]["y"], 5), round(candidate["location"]["x"], 5),
                      str(candidate["attributes"]["Subregion"]).strip().title(), candidate["address"]]
            # The locator also suggests numbers with the digits rearranged ("022-045-020")
            parcel_digits = "".join(char for char in candidate["address"].split(",")[0] if char.isdigit())
            if any(parcel_digits.startswith(reading) for reading in readings) and parcel not in found:
                found.append(parcel)
    return cache_put(key, found)


def zip_counties(*texts):
    """List the counties covered by the ZIP codes in the texts."""
    return {county for text in texts for zip_code in re.findall(r"\b9\d{4}\b", text)
            for county in ZIP_COUNTIES.get(zip_code, [])}


def nearest_parcel(parcel_text, points, counties):
    """Find the parcel with the listed number closest to the manifest's addresses and the dairy,
    if that number exists in one of the manifest's counties. It may lie across a county line."""
    for part in parcel_text.split(","):
        apn = normalize_apn(part)
        # Skip dummy APNs that only have 0s and 1s
        if not apn or set(apn) <= set("01-"):
            continue
        candidates = parcel_candidates(apn)
        if any(county in counties for _, _, county, _ in candidates):
            closest = min(candidates, key=lambda c: min(geodesic(c[:2], point).miles for point in points))
            return closest[0], closest[1]
    return None


def norm_addr(address):
    """Spell an address the way libpostal does. Return None for a PO Box, which is not a place."""
    cleaned = address.replace(": ", " ").strip().lower()
    if re.search(r"\bp\.?o\.?\s*box\b", cleaned):
        return None
    expansions = expand_address(cleaned, languages=["en"])
    return expansions[0] if expansions else cleaned


def geocode_address(address, county):
    """Find (lat, lng, notes) for a street-level address. The county only helps the search."""
    normalized = norm_addr(address)
    if not normalized:
        return None, None, None
    county_key = str(county).strip().lower()
    county_key = "" if county_key in ("nan", "none") else county_key
    key = cache_key(normalized, county_key)
    if key in cache:
        return cache[key]
    county_name = FOLDER_COUNTIES.get(county_key, county_key.title()) if county_key else None

    # Two crossing streets count only if ArcGIS finds the crossing itself, not a point on one street
    intersection = format_intersection(address, county_name)
    if intersection:
        hit = arcgis_geocoder(intersection)
        if hit and hit[3] == "StreetInt":
            return cache_put(key, (round(hit[1], 5), round(hit[2], 5), {"address": hit[0], "source": "arcgis"}))
        return cache_put(key, (None, None, {"source": "not_an_intersection"}))

    # Add the county, unless the address names its own city or ZIP
    tags = {tag for _, tag in parse_address(normalized)}
    names_place = tags & {"city", "postcode"}
    query = f"{address}, {county_name} County, CA" if county_name and not names_place else f"{address}, CA"
    for source, geocoder in [("arcgis", arcgis_geocoder), ("google", google_lookup)]:
        hit = geocoder(query)
        # Keep a hit only if it names a house number or road, not just a city
        if hit and any(tag in ("house_number", "road") for _, tag in parse_address(hit[0])):
            return cache_put(key, (round(hit[1], 5), round(hit[2], 5), {"address": hit[0], "source": source}))
    return cache_put(key, (None, None, {"source": "not_street_level"}))


def geocode_if_valid(address, county):
    """Find (lat, lng) for an address, or None when the address is blank or cannot be placed."""
    if not isinstance(address, str) or not address.strip():
        return None
    lat, lng = geocode_address(address, county)[:2]
    return None if lat is None else (lat, lng)


def row_text(row, param_key):
    """Get the row's value for a parameter as a stripped string, or "" when it is missing."""
    value = row[COLS[param_key]]
    return str(value).strip() if pd.notna(value) else ""


def resolve_destination_address(row, manual_coords, origin):
    """Find where one manifest's waste went. Return the destination text, the column it came
    from, and its (lat, lng), trying each source in turn until one can be placed."""
    destination_county = row_text(row, "destination_county")
    # Destination County is blank on most rows, so the dairy's folder is the hint for addresses
    county = destination_county or row["County"]
    parcel_raw, cross_raw, addr_raw, contact_raw, hauler_raw = (row_text(row, key) for key in [
        "destination_parcel_number", "destination_nearest_cross_street", "destination_address",
        "destination_contact_address", "hauler_address"])
    coords = COORD_RE.match(cross_raw)

    # The address plus its cross street and county, tried before the address on its own
    blob = " ".join(p for p in [addr_raw, "" if coords else cross_raw, destination_county] if p) if addr_raw else ""

    # A contact address needs some substance, and a hauler counts only if it looks like a farm or compost yard
    if sum(char.isalnum() for char in contact_raw) < 5:
        contact_raw = ""
    hauler_text = f"{row_text(row, 'hauler_name')} {hauler_raw}".lower()
    if not any(word in hauler_text for word in ("farm", "compost", "fertilizer")):
        hauler_raw = ""

    # The sources in the order tried below. The first non-blank one labels a manifest that cannot be placed.
    sources = [(parcel_raw, "destination_parcel_number"),
               (cross_raw if coords else "", "destination_nearest_cross_street"),
               (blob, "destination_address"),
               (cross_raw if not coords else "", "destination_nearest_cross_street"),
               (contact_raw, "destination_contact_address"),
               (hauler_raw, "hauler_address")]
    best = next(((text, COLS[key]) for text, key in sources if text), None)
    if best is None:
        return None, None, None
    if manual_coords is not None:  # a hand-entered coordinate wins
        return *best, manual_coords

    address_text = address_point = None
    for text in [blob] if blob == addr_raw else [blob, addr_raw]:
        address_point = geocode_if_valid(text, county)
        if address_point:
            address_text = text
            break
    contact_point = geocode_if_valid(contact_raw, county)

    # 1: parcel number, the match closest to the addresses or the dairy, in a county the manifest's ZIPs name
    points = [point for point in (address_point, contact_point, origin) if point]
    counties = zip_counties(row_text(row, "origin_dairy_address"), addr_raw, contact_raw)
    parcel_point = nearest_parcel(parcel_raw, points, counties) if points else None
    if parcel_point:
        return parcel_raw, COLS["destination_parcel_number"], parcel_point

    # 2: cross street holding written coordinates
    if coords:
        return cross_raw, COLS["destination_nearest_cross_street"], (float(coords.group(1)), float(coords.group(2)))

    # 3: destination address
    if address_point:
        return address_text, COLS["destination_address"], address_point

    # 3b: cross street, only when it names two streets (a single street is a line, not a point)
    if format_intersection(cross_raw):
        cross_point = geocode_if_valid(cross_raw, county)
        if cross_point:
            return cross_raw, COLS["destination_nearest_cross_street"], cross_point

    # 4: contact address, then 5: the address of a farm or compost yard hauler
    if contact_point:
        return contact_raw, COLS["destination_contact_address"], contact_point
    hauler_point = geocode_if_valid(hauler_raw, county)
    if hauler_point:
        return hauler_raw, COLS["hauler_address"], hauler_point
    return *best, None


def has_city_or_zip(address):
    """Check whether an address names a city or has a ZIP code."""
    return bool(re.search(r"\b9\d{4}\b", address)) or any(tag == "city" for _, tag in parse_address(address.lower()))


def backfill_origin_addresses(df):
    """Use the dairy summary's address for the report when the manifest's origin address is blank
    or has no city or ZIP (such as "26298 Groves Road"). The summary lists each report by PDF name."""
    col = COLS["origin_dairy_address"]
    summary = pd.read_csv(DATA_ROOT / "Dairy_Report_Summary_Region_5_2024_with_source_pdf.csv").dropna(
        subset=["Source PDF", "Dairy Address"])
    summary = dict(zip(summary["Source PDF"].map(lambda name: Path(name).stem), summary["Dairy Address"]))
    use_summary = ~df[col].fillna("").map(has_city_or_zip) & df["Source PDF"].isin(summary.keys())
    df.loc[use_summary, col] = df.loc[use_summary, "Source PDF"].map(summary)
    print(f"Took {use_summary.sum()} origin addresses from the dairy summary ({df[col].isna().sum()} still blank)")
    return df


def geocode_manifests(df):
    """Add origin and destination coordinates, and the destination text and its source column."""
    for key in ["destination_address_final", "destination_address_final_source",
                "origin_geo_lat", "origin_geo_lng", "destination_geo_lat", "destination_geo_lng"]:
        df[COLS[key]] = None
    for idx, row in df.iterrows():
        origin = geocode_if_valid(row[COLS["origin_dairy_address"]], row["County"])
        if origin:
            df.at[idx, COLS["origin_geo_lat"]], df.at[idx, COLS["origin_geo_lng"]] = origin
        lat, lng = row[COLS["latitude"]], row[COLS["longitude"]]
        manual_coords = (lat, lng) if pd.notna(lat) and pd.notna(lng) else None
        text, source, destination = resolve_destination_address(row, manual_coords, origin)
        if text:
            df.at[idx, COLS["destination_address_final"]] = text
            df.at[idx, COLS["destination_address_final_source"]] = source
        if destination:
            df.at[idx, COLS["destination_geo_lat"]], df.at[idx, COLS["destination_geo_lng"]] = destination
    return df


def backfill_quantities(df):
    """Fill manure tons from cubic yards and solids from moisture, and drop impossible solids."""
    # The form records density as "yards per ton" ("Manure Density (if amount reported in
    # cubic yards): 1.63 yards per ton"), so tons = cubic yards / density
    density = df[COLS["manure_density"]]
    for mass_col, volume_col in [(COLS["manure_amount"], COLS["manure_amount_yards"]),
                                 (COLS["manure_ton_per_haul"], COLS["manure_yard_per_haul"])]:
        needs_mass = df[mass_col].isna() & df[volume_col].notna()
        df.loc[needs_mass, mass_col] = df.loc[needs_mass, volume_col] / density[needs_mass].where(density > 0)
        print(f"  Backfilled {needs_mass.sum()} mass values for {mass_col}")
        df.drop(columns=volume_col, inplace=True)

    solids, moisture = COLS["manure_solids_percent"], COLS["manure_moisture_percent"]
    invalid = df[solids] > 100
    df.loc[invalid, solids] = np.nan
    from_moisture = df[solids].isna() & df[moisture].notna()
    df.loc[from_moisture, solids] = 100 - df.loc[from_moisture, moisture]
    print(f"  Nulled {invalid.sum()} solids values >100, and backfilled {from_moisture.sum()} from moisture")
    return df.drop(columns=[COLS["manure_density"], moisture])


def add_haul_estimates(df, waste_type):
    """Add a "Number of <bin> Hauls for Analysis" column for each truck size. A manifest without
    a haul size has its amount split between the sizes, in the proportion seen on manifests with one."""
    spec = WASTE_TYPES[waste_type]
    rate, amount = df[COLS[spec["rate"]]], df[COLS[spec["amount"]]]
    has_raw = rate.notna() & df[COLS[spec["hauls"]]].notna()
    n_hauls = pd.to_numeric(df[COLS[spec["hauls"]]], errors="coerce").round().astype("Int64")
    (lo, (lo_min, lo_max, lo_capacity)), (hi, (hi_min, hi_max, hi_capacity)) = spec["bins"].items()
    in_lo = rate.between(lo_min, lo_max, inclusive="left")
    in_hi = rate.between(hi_min, hi_max, inclusive="left")

    p_lo = amount[in_lo].sum() / (amount[in_lo].sum() + amount[in_hi].sum())
    print(f"  {waste_type}: {p_lo:.0%} of the amount with a reported haul size is in the {lo} bin")
    est_lo = (amount * p_lo / lo_capacity).round().astype("Int64")
    est_hi = (amount * (1 - p_lo) / hi_capacity).round().astype("Int64")
    df[f"Number of {lo} Hauls for Analysis"] = est_lo.where(~(has_raw & in_lo), n_hauls).where(~(has_raw & in_hi), 0)
    df[f"Number of {hi} Hauls for Analysis"] = est_hi.where(~(has_raw & in_hi), n_hauls).where(~(has_raw & in_lo), 0)


def std_dest_type(value):
    """Label a destination type as one or more of Broker, Composting Facility and Farmer."""
    if pd.isna(value) or not str(value).strip():
        return "Blank"
    text = str(value).lower()
    matched = [name for name, keywords in DEST_TYPE_MAP.items() if any(keyword in text for keyword in keywords)]
    return ", ".join(matched) if matched else "Other"


def prepare_manifests(df):
    """Standardize destination types, split into manure and wastewater records, and add haul estimates."""
    df[COLS["destination_type_std"]] = df[COLS["destination_type"]].apply(std_dest_type)
    manure = df.loc[df["Manifest Type"].isin(["manure", "both"]),
                    [col for col in df.columns if col not in SPECIFIC_COLS["wastewater"]]].copy()
    manure[COLS["is_trucked"]] = True
    wastewater = df.loc[df["Manifest Type"].isin(["wastewater", "both"]),
                        [col for col in df.columns if col not in SPECIFIC_COLS["manure"]]].copy()
    add_haul_estimates(manure, "manure")
    add_haul_estimates(wastewater, "wastewater")
    return manure, wastewater


def save_manifest_type(df, waste_type):
    """Write output_data/processed_<waste_type>_manifests.csv in report column order."""
    quantity_cols = [col for col in PARAMETERS_DF["parameter_name"]
                     if col in SPECIFIC_COLS[waste_type] and not col.startswith("Method Used")]
    columns = METADATA_COLS + [COLS[key] for key in [
        "origin_dairy_name", "origin_dairy_address", "origin_geo_lat", "origin_geo_lng",
        "destination_address_final", "destination_address_final_source", "destination_type_std",
        "destination_geo_lat", "destination_geo_lng", "haul_date_first", "haul_date_last",
        "is_pipeline", "is_trucked",
    ]] + quantity_cols + [col for col in df.columns if col.startswith("Number of")]
    out = df[[col for col in columns if col in df.columns]].copy()
    tons = [col for col in out.columns if "(tons)" in col]
    out[tons] = out[tons].round(1)  # backfilled tonnages would otherwise carry float noise
    out.to_csv(f"output_data/processed_{waste_type}_manifests.csv", index=False)
    print(f"Saved {len(out)} rows to processed_{waste_type}_manifests.csv")


OCR new PDFs and score the page detector (slow: OCR, and LLMWhisperer API calls)

In [ ]:
# Score the page detector against the manifest pages.
# "files" includes a row per PDF with manifest count and pages
detect_pages = False # set True to re-run detection on every report, which takes ~ 6 hours
saved = pd.read_csv(FILES_CSV, dtype=str, keep_default_na=False).set_index("filename")
ocr_paths = ocr_txt_paths()
rows = []
for county in COUNTIES:
    for pdf in (DATA_ROOT / REGION / county).glob("*/original/*"):
        if pdf.suffix.lower() != ".pdf":
            continue
        text = ocr_paths[pdf.stem].read_text(encoding="utf-8", errors="ignore") if pdf.stem in ocr_paths else ""
        manifests = identify_manifest_pages(text)
        row = {
            "county": county,
            "template": pdf.parents[1].name,
            "filename": pdf.name,
            "manifest_count": len(manifests),
            "manifest_pages": ",".join(str(page) for _, (first, last), _ in manifests
                                        for page in range(first, last + 1)),
            "detected_pages": "",
        }
        if pdf.name in saved.index:
            old = saved.loc[pdf.name]
            row.update(manifest_count=old["manifest_count"] or row["manifest_count"],
                        manifest_pages=old["manifest_pages"] or row["manifest_pages"],
                        detected_pages=old["detected_pages"])
        if detect_pages:
            row["detected_pages"] = ",".join(map(str, detect_manifest_pages_in_pdf(pdf)))
            print(f"  [{len(rows) + 1}] {pdf.name} page #s {row['detected_pages']}")
        rows.append(row)
files = pd.DataFrame(rows)
files.to_csv(FILES_CSV, index=False, lineterminator="\r\n")

def page_set(pages):
    """Turn "52,53,54" into {52, 53, 54}, and a blank cell into an empty set."""
    return {int(page) for page in pages.split(",")} if pages else set()


found = extra = missed = 0
for manifest_pages, detected_pages in zip(files["manifest_pages"], files["detected_pages"]):
    manifest, detected = page_set(manifest_pages), page_set(detected_pages)
    found += len(manifest & detected)
    extra += len(detected - manifest)
    missed += len(manifest - detected)
print(f"Detector: FP rate {extra / (found + extra):.2%}  FN rate {missed / (found + missed):.2%}"
      f"  ({found} TP, {extra} FP, {missed} FN)")

# OCR the reports that have no fitz or tesseract text yet. Slow, and may call the LLMWhisperer API.
for pdf in (DATA_ROOT / REGION).glob("*/*/original/*"):
    if pdf.suffix.lower() != ".pdf" or any(ocr_output_path(pdf, m).exists() for m in ["fitz", "tesseract"]):
        continue
    pages = detect_manifest_pages_in_pdf(pdf)
    print(f"{pdf.name}: manifest pages {pages}")
    if not pages:
        save_text(ocr_output_path(pdf, "fitz"), "no_manifests_found")
        continue
    tesseract, confidence = tesseract_text(pdf, pages)
    texts = {"fitz": fitz_text(pdf, pages), "tesseract": tesseract}
    # Low confidence, or either of these form markers, means handwriting that only LLMWhisperer reads
    if confidence < 60 or "R5-2013-0122" in tesseract.upper() or "CUBIC YARDS" in tesseract.upper():
        texts["llmwhisperer"] = llmwhisperer_text(pdf, pages)
    for method, text in texts.items():
        save_text(ocr_output_path(pdf, method), text)
    print(f"  Saved {', '.join(texts)} text")

Extract manifests from the OCR text

In [ ]:
# Extract every manifest record from the OCR text (LLMWhisperer's when there is one).
# To also rewrite the ~1,600 split manifest PDFs, run save_manifest_pdfs on each path.
records = []
for n, (stem, path) in enumerate(sorted(ocr_txt_paths().items()), start=1):
    records += extract_manifests_from_txt(path)
    if n % 50 == 0:
        print(f"Processed {n} files")

df = coerce_columns(pd.DataFrame(records))
has_manure, has_wastewater = df[COLS["manure_amount"]].notna(), df[COLS["wastewater_amount"]].notna()
df["Manifest Type"] = np.select([has_manure & has_wastewater, has_manure, has_wastewater],
                                ["both", "manure", "wastewater"], "unknown")
columns = [*COLS.values(), "Parameter Template", "Source PDF", "Start Page", "End Page",
           "Manifest Number", "County", "Manifest Type", "Form Template"]
df[columns].to_csv("output_data/all_manifests_as_written_automatic.csv", index=False)
print(f"Saved {len(df)} manifest records")

Process the automatic and validated manifests

In [ ]:
# Automatic records are processed only to score them against the validated ones (Fig. S10)
df_manure_auto, df_ww_auto = prepare_manifests(pd.read_csv("output_data/all_manifests_as_written_automatic.csv"))
print(f"Automatic: {len(df_manure_auto)} manure and {len(df_ww_auto)} wastewater records\n")

# Validated records, minus the rows validators marked as duplicates of another manifest.
# all_manifests_as_written_validated.csv is checked by hand: read it, never write it.
validated = coerce_columns(pd.read_csv("output_data/all_manifests_as_written_validated.csv"))
validated = validated[validated["Is Duplicate"] != "x"].reset_index(drop=True)
validated = backfill_quantities(geocode_manifests(backfill_origin_addresses(validated)))
df_manure_validated, df_ww_validated = prepare_manifests(validated)
save_manifest_type(df_manure_validated, "manure")
save_manifest_type(df_ww_validated, "wastewater")

Manure and wastewater haul size bins

In [ ]:
# One column per waste type. Pipeline discharges are not trucked, so they have no haul size.
panels = [
    ("manure", df_manure_validated, df_manure_validated, "tons", ("left", "right")),
    ("wastewater", df_ww_validated[df_ww_validated[COLS["is_pipeline"]].ne(True)], df_ww_validated,
     "gallons", ("right", "left")),
]
fig_hauls, axes = plt.subplots(2, 2, figsize=(12, 8.75))
for column, (waste_type, trucked, all_records, unit, label_sides) in enumerate(panels):
    spec = WASTE_TYPES[waste_type]
    rate_col, haul_col = COLS[spec["rate"]], COLS[spec["hauls"]]
    ax_hist, ax_facility = axes[0, column], axes[1, column]
    ax_totals = ax_hist.twinx()

    # Average haul size per facility, and how many hauls that facility reported
    reported = trucked.dropna(subset=[rate_col, haul_col])
    facility_hauls = reported.groupby("Source PDF").agg(avg_rate=(rate_col, "mean"), total_hauls=(haul_col, "sum"))
    counts, edges = np.histogram(facility_hauls["avg_rate"], bins=10)
    bin_totals = np.histogram(facility_hauls["avg_rate"], bins=edges, weights=facility_hauls["total_hauls"])[0]
    averages = [facility_hauls["avg_rate"].mean(), np.average(reported[rate_col], weights=reported[haul_col])]

    ax_hist.set_title(f"{waste_type.title()} Haul Size Distribution")
    ax_hist.hist(facility_hauls["avg_rate"], bins=edges, color=COLORS[waste_type])
    ax_totals.scatter((edges[:-1] + edges[1:]) / 2, bin_totals, color="black", s=64, zorder=3)
    for average, style, side in zip(averages, ["solid", "dotted"], label_sides):
        ax_hist.axvline(average, color="black", lw=2, ls=style)
        # Gallons are large whole numbers, but tons need one decimal
        label = str(int(average)) if unit == "gallons" else str(round(average, 1))
        ax_hist.annotate(label, xy=(average, 0.97), xycoords=("data", "axes fraction"), ha=side, va="top", fontsize=10)
    ax_hist.set_xlabel(f"{unit.capitalize()} per haul")
    ax_hist.set_ylabel("Number of facilities")
    ax_hist.set_ylim(0, counts.max() * 1.1)
    ax_totals.set_ylabel("Total hauls")
    ax_totals.set_ylim(0, bin_totals.max() * 1.1)

    # Each report's total exports and number of manifest records
    facility = all_records.groupby("Source PDF").agg(total_amount=(COLS[spec["amount"]], "sum"),
                                                     manifest_count=("Manifest Number", "nunique"))
    ax_facility.set_title(f"{waste_type.title()} Facility Exports")
    ax_facility.scatter(facility["total_amount"], facility["manifest_count"], color=COLORS[waste_type])
    ax_facility.set_xlabel(f"Total Facility Exports ({unit})")
    ax_facility.set_ylabel("Manifests per Facility")

# Legend entries are drawn as empty plots so one legend can describe both columns
legend_ax = axes[0, 1]
legend_ax.plot([], [], "s", color="black", ms=10, label="Facility Count")
legend_ax.plot([], [], "o", color="black", ms=8, label="Total Hauls in Bin")
legend_ax.plot([], [], "-", color="black", lw=2, label="Average by Facility")
legend_ax.plot([], [], ":", color="black", lw=2, label="Average by Hauls")
fig_hauls.legend(*legend_ax.get_legend_handles_labels(), loc="upper left", bbox_to_anchor=(1.2, 1),
                 bbox_transform=legend_ax.transAxes, fontsize=12)
fig_hauls.tight_layout()
fig_hauls.subplots_adjust(right=0.82)
fig_hauls.savefig("figures/Supplementary_Figure_2.png", dpi=200, bbox_inches="tight")

Validated vs automatic extraction accuracy (Fig. S10)

In [ ]:
def value_variants(text):
    """List every libpostal spelling of a value, keeping only letters and digits. Two values match if
    any spellings agree, so case, punctuation and abbreviations ("Ave." vs "Avenue") don't matter."""
    # expand_address("") returns [], so a blank becomes {""} and two blanks agree
    return {re.sub(r"[^a-z0-9]", "", s) for s in expand_address(text, languages=["en"])} or {""}


# Compare the extracted parameter fields only. Is Trucked is set to True for every manure
# record on both sides, so it would always score 100%.
skip_cols = {"Destination Type", "Destination Address Final Source", "Is Trucked"}
keys = ["Source PDF", "Manifest Number"]
stats, errors_by_field = [], {}
for validated_df, auto_df in [(df_manure_validated, df_manure_auto), (df_ww_validated, df_ww_auto)]:
    merged = validated_df.merge(auto_df, on=keys, suffixes=("_validated", "_auto"))
    for col in validated_df.columns:
        if (col not in COL_TYPES or col in skip_cols or col in errors_by_field
                or "(Geocoded)" in col or "Method Used" in col):
            continue
        compared = merged[merged[f"{col}_validated"].notna() | merged[f"{col}_auto"].notna()]
        agree = pd.Series([bool(value_variants(a) & value_variants(b)) for a, b in zip(
            compared[f"{col}_validated"].fillna("").astype(str), compared[f"{col}_auto"].fillna("").astype(str))],
            index=compared.index, dtype=bool)
        errors_by_field[col] = compared.loc[~agree, keys]
        stats.append({"field": col, "manual_count": validated_df[col].notna().sum(),
                      "auto_count": auto_df[col].notna().sum(), "accuracy": agree.mean() * 100})
stats = (pd.DataFrame(stats).query("manual_count >= 50 and auto_count >= 50")
         .sort_values("accuracy", ascending=False, kind="stable").reset_index(drop=True))

# Errors corrected per field, plus records with at least one correction. Result-impacting
# fields feed the analyses: haul dates, quantities, and locations.
IMPACTFUL = ("Date", "Address", "Parcel", "Cross Street", "(tons)", "(Gallons)", "Hauls")
errors = pd.DataFrame({"Field": stats["field"],
                       "Errors Corrected": [len(errors_by_field[field]) for field in stats["field"]]})
for label, fields in [("Manifests with at least one error", stats["field"]),
                      ("Manifests with a result-impacting correction",
                       [field for field in stats["field"] if any(key in field for key in IMPACTFUL)])]:
    errors.loc[len(errors)] = [label, len(pd.concat([errors_by_field[field] for field in fields]).drop_duplicates())]
errors.to_csv("output_data/extraction_errors_by_field.csv", index=False)
print(errors.to_string(index=False))
print(stats.to_string(index=False, float_format="%.2f"))

bar_width = 0.35
x = np.arange(len(stats))
fig_acc, ax_counts = plt.subplots(figsize=(12, 7))
ax_accuracy = ax_counts.twinx()
manual_bars = ax_counts.bar(x - bar_width / 2, stats["manual_count"], width=bar_width, alpha=0.7,
                            color=COLORS["manual"])
auto_bars = ax_counts.bar(x + bar_width / 2, stats["auto_count"], width=bar_width, alpha=0.7,
                          color=COLORS["extracted"])
accuracy_dots = ax_accuracy.plot(x, stats["accuracy"], "o", color=COLORS["accuracy"], markersize=8, zorder=5)[0]
ax_counts.set_ylabel("Count of Non-Null Values")
ax_counts.set_xticks(x)
ax_counts.set_xticklabels(stats["field"], rotation=45, ha="right")
ax_counts.legend([manual_bars, auto_bars, accuracy_dots], ["Manual Count", "Automatic Count", "Accuracy (%)"],
                 bbox_to_anchor=(0.7, 1.25), ncol=3, frameon=False)
ax_accuracy.set_ylabel("Accuracy (%)")
ax_accuracy.set_ylim(0, 100)
fig_acc.tight_layout()
fig_acc.savefig("figures/Figure_S10_Manifest_Extraction_Accuracy.png")

Manifests per form template (Table S1)

In [ ]:
# Manifests per form template. Each validated manifest takes the Form Template of the automatic
# manifest starting on the same page. "forms submitted" drops only exact copies and keeps CAG018001.
auto_templates = (pd.read_csv("output_data/all_manifests_as_written_automatic.csv",
                              usecols=["Source PDF", "Start Page", "Form Template"])
                  .drop_duplicates(["Source PDF", "Start Page"]))
manifests = (pd.read_csv("output_data/all_manifests_as_written_validated.csv")
             .drop_duplicates(["Source PDF", "Start Page"])
             .merge(auto_templates, on=["Source PDF", "Start Page"], how="left"))
kept = manifests["Is Duplicate"] != "x"
template_counts = pd.DataFrame({
    "analysis set": manifests.loc[kept, "Form Template"].value_counts(),
    "forms submitted": manifests.loc[kept | manifests["Form Template"].eq("CAG018001"), "Form Template"].value_counts(),
}).reindex(TEMPLATES_DF["template_key"]).fillna(0).astype(int)
template_counts["% of forms submitted"] = (100 * template_counts["forms submitted"]
                                           / template_counts["forms submitted"].sum()).round(1)
print(template_counts.to_string())
print(f"\nTotal: {template_counts['analysis set'].sum()} manifests analysed, "
      f"{template_counts['forms submitted'].sum()} forms submitted")